# 05 — Add a New Baseline Method

This notebook walks through the minimum steps to contribute a new baseline selection method. We add a toy baseline (`importance_sampling`) that draws with probability proportional to each point's squared norm, compare it to `baseline_uniform`, and point out where to register it for the rest of the pipeline.

For production contributions, follow the more detailed template in [CONTRIBUTING.md § How to Add a New Baseline](../CONTRIBUTING.md#how-to-add-a-new-baseline).

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'coreset_selection').is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt

## 1. The baseline protocol

Every baseline in this repo implements the same signature:

```python
def baseline_<name>(
    X: np.ndarray,      # features, shape (N, d)
    k: int,             # coreset size
    *,
    seed: int = 0,
    **kwargs,           # method-specific options
) -> np.ndarray         # integer indices, shape (k,)
```

Return value: **integer indices** (not a boolean mask) of `k` selected points.

## 2. Implement the toy baseline

In [ ]:
def baseline_importance_sampling(X: np.ndarray, k: int, *, seed: int = 0) -> np.ndarray:
    """Sample ``k`` rows of ``X`` with probability proportional to ‖x_i‖².

    Parameters
    ----------
    X : np.ndarray, shape (N, d)
        Feature matrix.
    k : int
        Target coreset size.
    seed : int
        Random seed.

    Returns
    -------
    np.ndarray, shape (k,)
        Selected indices (without replacement).
    """
    rng = np.random.default_rng(seed)
    weights = (X ** 2).sum(axis=1)
    weights = weights / weights.sum()
    return rng.choice(len(X), size=k, replace=False, p=weights)


# Quick sanity test on synthetic data
rng = np.random.default_rng(0)
X_toy = rng.normal(size=(1000, 10))
idx = baseline_importance_sampling(X_toy, k=30, seed=42)
assert idx.shape == (30,)
assert len(set(idx)) == 30
assert idx.min() >= 0 and idx.max() < 1000
print('Sanity checks passed.')
print('First 10 indices:', idx[:10])

## 3. Compare with existing uniform baseline

Both return `k` indices; the difference is only in which points they favour.

In [ ]:
from coreset_selection.baselines import baseline_uniform

idx_uniform = baseline_uniform(X_toy, k=30, seed=42)
idx_importance = baseline_importance_sampling(X_toy, k=30, seed=42)

norms = np.linalg.norm(X_toy, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].hist(norms[idx_uniform], bins=15, edgecolor='k'); axes[0].set_title('uniform — norm of selected')
axes[1].hist(norms[idx_importance], bins=15, edgecolor='k', color='tab:orange'); axes[1].set_title('importance — norm of selected')
for ax in axes:
    ax.set_xlabel('‖x_i‖')
plt.tight_layout()
plt.show()

## 4. Steps to make it a proper contribution

To turn this prototype into a first-class baseline in the repository:

1. **Create** `coreset_selection/baselines/importance_sampling.py` and paste the function.
2. **Also implement the quota-constrained variant** `baseline_importance_sampling_quota(X, k, geo, *, seed=0)` — see existing modules like `uniform.py` for the per-group pattern.
3. **Register** in `coreset_selection/baselines/__init__.py`:
   ```python
   from .importance_sampling import baseline_importance_sampling, baseline_importance_sampling_quota
   __all__ += ['baseline_importance_sampling', 'baseline_importance_sampling_quota']
   BASELINE_METHODS['importance_sampling'] = baseline_importance_sampling
   BASELINE_METHODS['importance_sampling_quota'] = baseline_importance_sampling_quota
   ```
4. **Test** in `coreset_selection/tests/test_importance_sampling.py` (shape, uniqueness, seed reproducibility).
5. **Document** by adding an entry to [docs/api/baselines.md](../docs/api/baselines.md) following the format of existing entries.
6. **Benchmark** against the existing 8 methods: run `scripts/launchers/run_baselines.py`, then `scripts/analysis/evaluate_coresets.py`, then `scripts/analysis/championship.py` — the new method automatically appears in the ranking.

## Cross-references

- [CONTRIBUTING.md](../CONTRIBUTING.md) — full PR workflow and style guide.
- [docs/api/baselines.md](../docs/api/baselines.md) — every existing baseline documented in the same style you should follow.